# Checks — order and risk

$$\mathrm{can\_start}(s) \iff \mathrm{Pred}(s) \subseteq C \land (\mathrm{risk}(s) \neq \mathrm{high} \lor H)$$

`is_allowed` = predecessors only. `can_start` adds the hands-clear gate and returns reasons.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))

from shopfloor import ShopContext, WorkOrder, can_start, complete_step, is_allowed, parse_sop

steps = parse_sop(root / "docs/sop_samples/miam_assembly_subset.md")
step6 = steps["STEP-06"]
step6

In [ ]:
order = WorkOrder(
    id="WO-310",
    completed={f"STEP-{i:02d}" for i in range(1, 6)},
)

print("order only:", is_allowed(step6, order))

ok, reasons = can_start(step6, order, ShopContext(hands_clear=False))
print("busy:", ok, reasons)

ok, reasons = can_start(step6, order, ShopContext(hands_clear=True))
print("clear:", ok, reasons)

In [ ]:
order = WorkOrder(id="WO-311")
ctx = ShopContext(hands_clear=False)

for sid in [f"STEP-{i:02d}" for i in range(1, 6)]:
    ok, _ = complete_step(order, steps[sid], ctx)
    assert ok, sid

ok, reasons = complete_step(order, step6, ctx)
print("blocked:", ok, reasons)

ctx.hands_clear = True
ok, reasons = complete_step(order, step6, ctx)
print("after clear:", ok, sorted(order.completed))